# Proposal indicator analysis - walkthrough of the pipeline outputs

This notebook is a thin reader of what the ESGReveal-style pipeline (`pipeline/`) produced. It makes **no LLM calls**
and runs on the CSV files in `outputs/latest/` (written by `python -m pipeline.run run --all --confirm-full-run`).

Steps (see the README): `build-metadata` -> `build-kb --all` -> `score-pairs --all` -> `run --pilot` / `run --all --confirm-full-run`
-> `export-review` -> the Streamlit app. Method: `docs/method.md`; results: `docs/phase_3_report.md`.

In [ ]:
from pathlib import Path

import pandas as pd

from pipeline import config
from pipeline.agent.outputs import document_report_markdown

LATEST = config.LATEST_DIR
if not (LATEST / "document_summary.csv").exists():
    raise SystemExit(f"No pipeline run found under {LATEST}. Run: python -m pipeline.run run --pilot --copy-latest (or --all --confirm-full-run)")

summary = pd.read_csv(LATEST / "document_summary.csv", dtype=str, keep_default_na=False)
doc_ind = pd.read_csv(LATEST / "document_indicators.csv", dtype=str, keep_default_na=False)
matrix = pd.read_csv(LATEST / "disclosure_matrix.csv", dtype=str, keep_default_na=False)
novel = pd.read_csv(LATEST / "novel_indicators.csv", dtype=str, keep_default_na=False)
for c in ("total_indicators", "verbatim", "close", "none", "pct_matched", "llm_calls", "cost_usd"):
    summary[c] = pd.to_numeric(summary[c], errors="coerce")
print((LATEST / "LATEST_RUN.txt").read_text().splitlines()[0], "-", len(summary), "documents,", len(doc_ind), "document indicators,", len(matrix), "document x IndiKit matches,", len(novel), "novel")
summary[["relative_path", "country", "total_indicators", "verbatim", "close", "none", "pct_matched", "note"]].head(20)

## One document: the three-column table MEAL staff use

In [ ]:
from IPython.display import Markdown

doc_id = summary.sort_values("total_indicators", ascending=False)["doc_id"].iloc[0]   # pick any doc_id from `summary`
Markdown(f"### {summary.set_index('doc_id').loc[doc_id, 'relative_path']}\n\n" + document_report_markdown(doc_id, doc_ind, summary))

## Portfolio charts

In [ ]:
import matplotlib.pyplot as plt

by_country = pd.read_csv(LATEST / "portfolio_by_country.csv")
most_used = pd.read_csv(LATEST / "portfolio_most_used_indikit.csv", dtype={"indikit_id": str}).head(20)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
if len(by_country):
    by_country.set_index("country")[["verbatim", "close", "none"]].plot(kind="bar", stacked=True, ax=axes[0], color=["#2b8cbe", "#a6bddb", "#d9d9d9"])
    axes[0].set_title("Document indicators per country (verbatim / close / no IndiKit match)")
    axes[0].set_ylabel("document indicators")
if len(most_used):
    labels = most_used["indikit_name"] + " (" + most_used["indikit_id"] + ")"
    axes[1].barh(labels[::-1], most_used["documents"][::-1], color="#2b8cbe")
    axes[1].set_title("Most-used IndiKit indicators (documents)")
plt.tight_layout()
plt.show()

Markdown((LATEST / "portfolio_summary.md").read_text(encoding="utf-8"))

## Evaluation (when `python -m pipeline.run experiments` has been run on verified gold)

In [ ]:
results_md = config.EVAL_OUTPUT_DIR / "results.md"
if results_md.exists():
    display(Markdown(results_md.read_text(encoding="utf-8")))
    fig_path = config.EVAL_OUTPUT_DIR / "figures" / "ablation.png"
    if fig_path.exists():
        from IPython.display import Image
        display(Image(filename=str(fig_path)))
else:
    print("no evaluation results yet:", results_md, "- the experiments need verified gold documents (pipeline/eval/gold/).")

## Review loop

`python -m pipeline.run export-review` writes `outputs/latest/review.xlsx` (rows with `needs_review = true`, one sheet per country).
After MEAL staff fill in `reviewer_decision`, `python -m pipeline.run import-review --file outputs/latest/review.xlsx --export-gold`
applies the decisions (`document_indicators_reviewed.csv`), proposes metadata overrides as a separate diff and grows the gold set.